# LongFlow — score the split-path probe on GPU

Runtime: **any GPU**. Reads `splitpath_eval.zip` from Drive root. Scores the
four closed-loop renders and compares against last night's fixed reference
rows (cleanabl plain and both-polished).


In [ ]:
# ===== COLD START — run me first, wait for READY =====
NOTEBOOK_VERSION = "Score split-path (GPU) v1.0 (2026-08-18)"
print(f"*** {NOTEBOOK_VERSION} ***")
!pip install -q faster-whisper jiwer whisper-normalizer speechbrain praat-parselmouth

import torch
assert torch.cuda.is_available(), "no GPU — pick a GPU runtime"
import glob, json, os, sys, zipfile
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1
from src.eval.metrics import _ecapa, _normalizer, _whisper

from google.colab import drive
drive.mount("/content/drive")
candidates = glob.glob("/content/drive/MyDrive/splitpath_eval*.zip")  # root only
assert candidates, "no splitpath_eval*.zip in Drive root"
zip_path = candidates[0]
print(f"using {zip_path}")
AUD = "/content/eval_audio"
os.makedirs(AUD, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(AUD)
print(f"extracted {len(os.listdir(AUD))} entries")
print("READY")


In [ ]:
# ===== Closed-loop rows vs last night's references =====
import jiwer
import torchaudio.functional as taf

with open(f"{AUD}/splitpath_report.json") as f:
    report = json.load(f)
results = {"closed_loop": {}, "reference": {
    "cleanabl_plain_s0": {"wer": 0.116, "voice": 24.8, "sim": 0.474},
    "cleanabl_both_polish2_s0": {"wer": 0.243, "voice": 62.0, "sim": 0.522},
}}
WIN, HOP = 4.0, 2.0
ecapa = _ecapa("cuda")
n = _normalizer()
script_words = []
for line in report["cl_script"].splitlines():
    if ":" in line:
        line = line.split(":", 1)[1]
    script_words.append(line.strip())
SCRIPT = n(" ".join(w for w in script_words if w))
N_SCRIPT = len(SCRIPT.split())

def win_embs(path):
    x, sr = sf.read(path, dtype="float32")
    x16 = taf.resample(torch.from_numpy(x), sr, 16000).numpy()
    dur = len(x) / sr
    E, ts = [], []
    for i in range(int((dur - WIN) // HOP) + 1):
        seg = x16[int(i * HOP * 16000): int((i * HOP + WIN) * 16000)]
        if len(seg) < 16000:
            break
        emb = ecapa.encode_batch(torch.from_numpy(seg)[None].to("cuda"))[0, 0]
        E.append(emb.detach().cpu())
        ts.append(i * HOP)
    return torch.stack(E), np.array(ts), dur

def transcribe(path):
    segs, _ = _whisper("cuda").transcribe(str(path), language="en", beam_size=1)
    return " ".join((s.text or "").strip() for s in segs)

ref_E, _, _ = win_embs(f"{AUD}/t1_turnsplit_p0.wav")
ref = ref_E.median(0).values

def score(path):
    E, ts, dur = win_embs(path)
    sim = torch.nn.functional.cosine_similarity(E, ref[None], dim=-1).numpy()
    voice = sim >= 0.5
    horizon = 0.0
    for i in range(len(ts)):
        if voice[i]:
            horizon = ts[i] + WIN
    hyp = n(transcribe(path))
    nw = len(hyp.split())
    return {"duration_s": round(dur, 1),
            "wer_vs_script": round(jiwer.wer(SCRIPT, hyp) if hyp else 1.0, 3),
            "coverage_pct": round(100 * min(nw, N_SCRIPT) / N_SCRIPT, 1),
            "voice_pct": round(100 * float(voice.mean()), 1),
            "horizon_s": float(horizon),
            "sim_median": round(float(np.median(sim)), 3),
            "sim_final_third": round(float(np.median(sim[-max(1, len(sim) // 3):])), 3),
            "rate_wpm": round(60 * nw / dur, 1)}

for p in sorted(glob.glob(f"{AUD}/closed_loop/*.wav")):
    tag = os.path.basename(p)[:-4]
    results["closed_loop"][tag] = score(p)
    print(f"{tag}: {results['closed_loop'][tag]}", flush=True)

c = results["closed_loop"]
print("\n--- attribution read ---")
if "sp_audio_k3_s0" in c:
    r = c["sp_audio_k3_s0"]
    print(f"audio-only: WER {r['wer_vs_script']} (plain was 0.116 — should be ~equal), "
          f"sim {r['sim_median']} (texture gain is for the EAR, sim may not move)")
if "sp_feed_k2_s0" in c:
    r = c["sp_feed_k2_s0"]
    print(f"feedback-only: sim {r['sim_median']} vs plain 0.474 vs both-polish 0.522 — "
          f"did the identity lift come from the loop side?  WER {r['wer_vs_script']} vs 0.116")
if "sp_both_k2_ema_s0" in c:
    r = c["sp_both_k2_ema_s0"]
    print(f"both+EMA-noise: WER {r['wer_vs_script']} vs last night's 0.243 — "
          f"did correlated noise reclaim content?")
for st in ("sp_stack_s0", "sp_stack_s1"):
    if st in c:
        r = c[st]
        ok = r["wer_vs_script"] <= 0.06 and r["sim_median"] >= 0.50
        print(f"STACK {st}: WER {r['wer_vs_script']}, sim {r['sim_median']}, "
              f"voice {r['voice_pct']}% -> {'BARS MET' if ok else 'below bars'} "
              f"(need WER<=0.06 AND sim>=0.50 both seeds)")

with open("/content/drive/MyDrive/splitpath_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nmetrics on Drive root: splitpath_metrics.json")
print("Listening: sp_audio_k3_s0 (the best-sounding-today candidate) + knobfix k3 A/B vs old k3.")
